In [ ]:
import random
import time

from ugot import ugot

got = ugot.UGOT()
got.initialize("192.168.88.1")
got.show_light_rgb([1, 2, 3], 255, 255, 255)
got.play_audio_tts("Connection successful")

In [ ]:
got.load_models(['word_recognition'])

In [ ]:
while True:
    result = got.get_words_result()
    if result:
        if result == "LEFT":
            got.mecanum_turn_speed_times(2, 45, 90, 2)
        elif result == "RIGHT":
            got.mecanum_turn_speed_times(3, 45, 90, 0)
        elif result == "FORWARD":
            got.mecanum_move_speed(0, 20)
        elif result == "BACKWARD":
            got.mecanum_move_speed(1, 20)
    else:
        got.mecanum_stop()

In [ ]:
def play_wordle():
    word_bank = ["APPLE", "BEACH", "CHIPS", "DANCE", "EAGLE", "FLAME", "GRAPE", "HOUSE", "LIGHT", "PLANT"]
    target_word = random.choice(word_bank)
    
    # ANSI color codes for terminal feedback
    GREEN = "\033[42m\033[30m"   # Green background, black text
    YELLOW = "\033[43m\033[30m"  # Yellow background, black text
    GRAY = "\033[47m\033[30m"    # Gray background, black text
    RESET = "\033[0m"            # Reset colors back to normal
    
    attempts = 6
    for attempt in range(1, attempts + 1):
        # Get valid user input
        while True:
            guess = input(f"\nAttempt {attempt}/{attempts}. Enter your guess: ").upper()
            if len(guess) == 5 and guess.isalpha():
                break
            print("Invalid input. Please enter exactly 5 alphabetical letters.")

        # Build the colored feedback string
        feedback = []
        for i in range(5):
            if guess[i] == target_word[i]:
                feedback.append(f"{GREEN} {guess[i]} {RESET}")
            elif guess[i] in target_word:
                feedback.append(f"{YELLOW} {guess[i]} {RESET}")
            else:
                feedback.append(f"{GRAY} {guess[i]} {RESET}")
        
        print("Feedback: " + "".join(feedback))

        # Check for win condition
        if guess == target_word:
            print(f"\n🎉 Congratulations! You guessed the word '{target_word}' correctly!")
            return

    # Loss condition
    print(f"\nGame Over! You've run out of tries. The word was '{target_word}'.")

if __name__ == "__main__":
    play_wordle()

In [ ]:
import cv2
import numpy as np

# ANSI color codes for terminal feedback
GREEN = "\033[42m\033[30m"   # Green background, black text
YELLOW = "\033[43m\033[30m"  # Yellow background, black text
GRAY = "\033[47m\033[30m"    # Gray background, black text
RESET = "\033[0m"            # Reset colors back to normal

word_bank = ["APPLE", "RIGHT"]
target_word = random.choice(word_bank)
attempts = 6

got.open_camera()
try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()

        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # Get text recognition result
        result = got.get_words_result()

        if result and attempts > 0:
            result = result.upper()

            # Build the colored feedback string
            feedback = []
            for i in range(5):
                if result[i] == target_word[i]:
                    feedback.append(f"{GREEN} {result[i]} {RESET}")
                elif result[i] in target_word:
                    feedback.append(f"{YELLOW} {result[i]} {RESET}")
                else:
                    feedback.append(f"{GRAY} {result[i]} {RESET}")

            print("Feedback: " + "".join(feedback))

            # Check for win condition
            if result == target_word:
                got.play_audio_tts(f"Congratulations! You guessed the word '{target_word}' correctly!")
                break

            attempts -= 1

            # Check for lose condition
            if attempts <= 0:
                got.play_audio_tts("No attempts left. You lose!")
                break

            time.sleep(10)
            
        # Display the frame
        cv2.imshow("Text Recognition", img)

        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Closing camera window...")
            break

finally:
    cv2.destroyAllWindows()
    cv2.waitKey(1)